# DiD-BCF — D_contamination (linearity_degree=1)

**Workstream D · staggered adoption (cohort x event-time effects)**

stronger dynamics -> larger TWFE contamination

Fits DiD-BCF on the `D_contamination` scenario at `linearity_degree=1` and reports
metrics for **both** the plain DiD-BCF posterior and the proposed **posterior
correction** (Algorithm 1 of the theory note), so the correction can be judged
directly. Panel: N=200, 4 pre + 4 post periods.

> **Colab:** upload just this notebook and *Run all* — the first cell installs the
> dependencies and the second clones the engine automatically.

In [ ]:
# Colab: install the DiD-BCF dependencies (stochtree provides the BCF sampler).
%pip install -q stochtree scikit-learn joblib tqdm pandas numpy

In [ ]:
import os, sys

# --- Locate the DiD-BCF engine ------------------------------------------------
# So you can upload just THIS notebook to Colab and Run all. Resolution order:
#   1. `did_bcf_revision` already importable;
#   2. running inside a repo checkout (the parent folder holds the package);
#   3. otherwise clone https://github.com/hugogobato/DiD-BCF and use it.
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
ENGINE_SUBDIR = os.path.join("DiD-BCF", "Simulation_Studies_Revision")

def _locate_root():
    try:
        import did_bcf_revision  # noqa: F401
        return os.path.dirname(os.path.dirname(did_bcf_revision.__file__))
    except Exception:
        pass
    parent = os.path.abspath(os.path.join(os.getcwd(), ".."))
    if os.path.isdir(os.path.join(parent, "did_bcf_revision")):
        return parent
    if not os.path.isdir("DiD-BCF"):
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return os.path.abspath(ENGINE_SUBDIR)

ROOT = _locate_root()
sys.path.insert(0, ROOT)
print("Using DiD-BCF engine at:", ROOT)

from did_bcf_revision.runner import run_named
from did_bcf_revision.metrics import (compute_metrics, plain_vs_corrected,
                                      surface_metrics)

In [ ]:
REPS = 100      # replications (lower for a quick smoke test)
JOBS = 1        # parallel reps (keep 1 on a single-core/GPU Colab)

bcf_params = dict(num_gfr=50, num_mcmc=500, keep_every=5, num_chains=3)

summaries = run_named(
    "D_contamination",
    linearity_degree=1,
    reps=REPS,
    jobs=JOBS,
    bcf_params=bcf_params,
    prop_method="logit",   # pilot propensity for the posterior correction
    n_splits=2,            # cross-fitting folds for the correction
)
summaries.head()

In [ ]:
# Decomposed metrics: bias, MC SD/variance, RMSE, MAE, MAPE, coverage 90/95,
# interval length, calibration ratio (avg_post_sd/emp_sd), size/power and their
# Monte-Carlo SEs -- for plain AND corrected DiD-BCF.
metrics = compute_metrics(summaries)
plain_vs_corrected(metrics)

## CATT-surface metrics (the paper's headline RMSE/MAE/MAPE)

Within-replication RMSE/MAE/MAPE over the *individual* treated observations
(mean +/- SD across runs) plus the *pointwise* CATT coverage -- the evidence
that DiD-BCF recovers the heterogeneous effect that GATT-only methods cannot.

In [ ]:
surface_metrics(summaries)

## Goodman-Bacon decomposition (TWFE contamination)

How much of a naive TWFE estimate on this DGP comes from the
"already-treated as control" comparisons that bias it.

In [ ]:
from did_bcf_revision.dgps import generate_staggered_did
from did_bcf_revision.goodman_bacon import bacon_summary

df0 = generate_staggered_did(seed=0, linearity_degree=1)
bacon_summary(df0)